# Segmentation and Object Detection Studies

Portfolio source extract. See the repository README and PROVENANCE.md for methods, data requirements, original evidence and publication changes. Run from the notebooks/ directory.

In [ ]:
# Run this cell only once in a fresh local Jupyter environment
# Install ../requirements.txt in the selected environment before running.
# Publication: install dependencies from ../requirements.txt first.



In [ ]:
import json
import multiprocessing as mp
import os
import random
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
gpu_vram_gb = None

if use_amp:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    torch.backends.cudnn.benchmark = True
    print(f"Using device: {device} ({gpu_name}, {gpu_vram_gb:.1f} GB VRAM)")
else:
    print("CUDA is not available. Training will run on CPU.")



In [ ]:
ZIP_PATH = Path("../data/Segmentation Dataset.zip")
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "local_data" / "segmentation_dataset"
SAVE_DIR = PROJECT_ROOT / "runs" / "unet_segmentation"
IMAGE_SIZE = 512
RUN_TAG = "unet_original_crop_512_v1"
CHECKPOINT_PATH = SAVE_DIR / f"{RUN_TAG}_best_weights.pth"
HISTORY_PATH = SAVE_DIR / f"{RUN_TAG}_training_history.json"
HISTORY_PLOT_PATH = SAVE_DIR / f"{RUN_TAG}_training_history.png"
SAMPLE_PLOT_DIR = SAVE_DIR / f"{RUN_TAG}_segmentation_plots"

DATASET_ROOT = DATA_DIR / "Unet Dataset"
IMAGE_DIR = DATASET_ROOT / "RGB"
MASK_DIR = DATASET_ROOT / "MASKS"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Dataset zip not found: {ZIP_PATH}")

if not IMAGE_DIR.exists() or not MASK_DIR.exists():
    print("Extracting dataset zip...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print("Dataset extraction complete.")

if not IMAGE_DIR.exists() or not MASK_DIR.exists():
    raise FileNotFoundError(f"Expected extracted folders at {IMAGE_DIR} and {MASK_DIR}")

print(f"Dataset root: {DATASET_ROOT}")
print(f"Results will be saved to: {SAVE_DIR}")


In [ ]:
#Defining a Unet Class
class Unet(nn.Module):
  def __init__(self, num_classes):
      super(Unet, self).__init__()

      self.conv1 = self.tripleconv(3,64)
      self.pool1 = nn.MaxPool2d(2,2)

      self.conv2 = self.tripleconv(64,128)
      self.pool2 = nn.MaxPool2d(2,2)

      self.conv3 = self.tripleconv(128,256)
      self.pool3 = nn.MaxPool2d(2,2)

      self.conv4 = self.tripleconv(256,512)
      self.pool4 = nn.MaxPool2d(2,2)

      self.conv5 = self.tripleconv(512,1024)

      self.upconv1 = self.upconv(1024,512)
      self.conv6 = self.tripleconv(1024,512)

      self.upconv2 = self.upconv(512,256)
      self.conv7 = self.tripleconv(512,256)

      self.upconv3 = self.upconv(256,128)
      self.conv8 = self.tripleconv(256,128)

      self.upconv4 = self.upconv(128,64)

      self.conv9 = self.tripleconv(128,64)

      self.conv10 = self.singleconv(64, num_classes)


  def forward(self,x):

      c1 = self.conv1(x)
      p1 = self.pool1(c1)

      c2 = self.conv2(p1)
      p2 = self.pool2(c2)

      c3 = self.conv3(p2)
      p3 = self.pool3(c3)

      c4 = self.conv4(p3)
      p4 = self.pool4(c4)

      c5 = self.conv5(p4)

      u1 = self.upconv1(c5)
      cropped_c4 = self.crop(c4,u1.shape[2:])
      u1 = torch.cat([u1,cropped_c4],dim=1)
      u1 = self.conv6(u1)

      u2 = self.upconv2(u1)
      cropped_c3 = self.crop(c3,u2.shape[2:])
      u2 = torch.cat([u2,cropped_c3],dim=1)
      u2 = self.conv7(u2)

      u3 = self.upconv3(u2)
      cropped_c2 = self.crop(c2,u3.shape[2:])
      u3 = torch.cat([u3,cropped_c2],dim=1)
      u3 = self.conv8(u3)

      u4 = self.upconv4(u3)
      cropped_c1 = self.crop(c1,u4.shape[2:])
      u4 = torch.cat([u4,cropped_c1],dim=1)
      u4 = self.conv9(u4)

      out = self.conv10(u4)


      return out

  def tripleconv(self,in_channel,out_channel):
    return nn.Sequential(
        nn.Conv2d(in_channel,out_channel,3,padding=0),
        nn.ReLU(),
        nn.Conv2d(out_channel,out_channel,3,padding=0),
        nn.ReLU(),
    )


  def crop(self,x,size):
    return torchvision.transforms.functional.center_crop(x,size)


  def upconv(self,in_channel,out_channel):
    return nn.ConvTranspose2d(in_channel, out_channel, kernel_size=2, stride=2, padding=0, output_padding=0)


  def singleconv(self,in_channel,out_channel):
    return nn.Conv2d(in_channel,out_channel,1,padding=0)


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(
        self,
        image_dir,
        mask_dir,
        image_list,
        target_size=(512, 512),
        is_train=False,
        image_mean=None,
        image_std=None,
    ):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.images = list(image_list)
        self.target_size = target_size
        self.is_train = is_train
        self.image_mean = np.array(image_mean if image_mean is not None else [0.0, 0.0, 0.0], dtype=np.float32)
        self.image_std = np.array(image_std if image_std is not None else [1.0, 1.0, 1.0], dtype=np.float32)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = self.image_dir / img_name
        mask_name = Path(img_name).with_suffix(".png").name
        mask_path = self.mask_dir / mask_name

        image = cv2.imread(str(img_path))
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise FileNotFoundError(f"Image not found or unreadable: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"Mask not found or unreadable: {mask_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.is_train:
            if random.random() > 0.5:
                image = cv2.flip(image, 1)
                mask = cv2.flip(mask, 1)
            if random.random() > 0.5:
                angle = random.uniform(-6.0, 6.0)
                center = (image.shape[1] / 2.0, image.shape[0] / 2.0)
                rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
                image = cv2.warpAffine(
                    image,
                    rotation_matrix,
                    (image.shape[1], image.shape[0]),
                    flags=cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_REFLECT_101,
                )
                mask = cv2.warpAffine(
                    mask,
                    rotation_matrix,
                    (mask.shape[1], mask.shape[0]),
                    flags=cv2.INTER_NEAREST,
                    borderMode=cv2.BORDER_REPLICATE,
                )

            if random.random() > 0.6:
                alpha = 1.0 + random.uniform(-0.1, 0.1)
                beta = random.uniform(-8.0, 8.0)
                image = np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

            if random.random() > 0.6:
                gamma = random.uniform(0.85, 1.15)
                image = np.clip(
                    255.0 * np.power(image.astype(np.float32) / 255.0, gamma),
                    0,
                    255,
                ).astype(np.uint8)

        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        image = image.astype(np.float32) / 255.0
        image = (image - self.image_mean) / self.image_std
        image = torch.tensor(image.copy()).permute(2, 0, 1).float()
        mask = torch.tensor(mask.copy()).long()

        return image, mask



In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if use_amp:
    torch.cuda.manual_seed_all(42)

all_images = sorted([name for name in os.listdir(IMAGE_DIR) if name.lower().endswith(".jpg")])
random.shuffle(all_images)

train_size = int(0.7 * len(all_images))
val_size = int(0.15 * len(all_images))
train_list = all_images[:train_size]
val_list = all_images[train_size : train_size + val_size]
test_list = all_images[train_size + val_size :]

with torch.inference_mode():
    SUPERVISED_SIZE = tuple(Unet(num_classes=3)(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)).shape[-2:])
print(f"Model output size: {SUPERVISED_SIZE} from input {(IMAGE_SIZE, IMAGE_SIZE)}")

def compute_image_stats(image_dir, image_list, target_size):
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    total_pixels = 0

    for img_name in image_list:
        image = cv2.imread(str(Path(image_dir) / img_name))
        if image is None:
            raise FileNotFoundError(f"Image not found while computing stats: {img_name}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
        pixels = image.astype(np.float32).reshape(-1, 3) / 255.0
        channel_sum += pixels.sum(axis=0)
        channel_sq_sum += np.square(pixels).sum(axis=0)
        total_pixels += pixels.shape[0]

    image_mean = channel_sum / total_pixels
    image_std = np.sqrt(np.clip(channel_sq_sum / total_pixels - image_mean ** 2, a_min=1e-8, a_max=None))
    return image_mean.astype(np.float32), image_std.astype(np.float32)

def compute_class_weights(mask_dir, image_list, crop_size, num_classes=3):
    pixel_counts = torch.zeros(num_classes, dtype=torch.float64)

    for img_name in image_list:
        mask_name = Path(img_name).with_suffix(".png").name
        mask = cv2.imread(str(Path(mask_dir) / mask_name), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask not found while computing class weights: {mask_name}")
        mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = torchvision.transforms.functional.center_crop(torch.from_numpy(mask), crop_size).numpy()

        values, counts = np.unique(mask, return_counts=True)
        for value, count in zip(values, counts):
            if 0 <= int(value) < num_classes:
                pixel_counts[int(value)] += int(count)

    inv_freq = pixel_counts.sum() / pixel_counts.clamp_min(1)
    class_weights = inv_freq / inv_freq.mean()
    class_ratios = pixel_counts / pixel_counts.sum()
    return class_weights.float(), class_ratios.float()

image_mean, image_std = compute_image_stats(IMAGE_DIR, train_list, target_size=(IMAGE_SIZE, IMAGE_SIZE))

train_dataset = SegmentationDataset(
    IMAGE_DIR,
    MASK_DIR,
    train_list,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    is_train=True,
    image_mean=image_mean,
    image_std=image_std,
)
val_dataset = SegmentationDataset(
    IMAGE_DIR,
    MASK_DIR,
    val_list,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    is_train=False,
    image_mean=image_mean,
    image_std=image_std,
)
test_dataset = SegmentationDataset(
    IMAGE_DIR,
    MASK_DIR,
    test_list,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    is_train=False,
    image_mean=image_mean,
    image_std=image_std,
)

class_weights, class_ratios = compute_class_weights(MASK_DIR, train_list, SUPERVISED_SIZE, num_classes=3)
loss_class_weights = class_weights.clone()
loss_class_weights = loss_class_weights / loss_class_weights.mean()

batch_size = 16
eval_batch_size = 16

pin_memory = use_amp
requested_workers = min(8, os.cpu_count() or 1)

try:
    if requested_workers > 0:
        probe_lock = mp.get_context("spawn").Lock()
        del probe_lock
except PermissionError:
    requested_workers = 0
    print("Falling back to num_workers=0 because multiprocessing is unavailable in this environment.")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=requested_workers,
    pin_memory=pin_memory,
    persistent_workers=requested_workers > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=requested_workers,
    pin_memory=pin_memory,
    persistent_workers=requested_workers > 0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=requested_workers,
    pin_memory=pin_memory,
    persistent_workers=requested_workers > 0,
)

print(
    f"Total: {len(all_images)} | Train: {len(train_dataset)} | "
    f"Val: {len(val_dataset)} | Test: {len(test_dataset)}"
)
print(
    f"Train batch size: {batch_size} | Eval batch size: {eval_batch_size} | "
    f"Workers: {requested_workers} | AMP enabled: {use_amp}"
)
print(f"Image normalization mean: {[round(float(x), 4) for x in image_mean.tolist()]}")
print(f"Image normalization std: {[round(float(x), 4) for x in image_std.tolist()]}")
print(f"Train class pixel ratios: {[round(x, 4) for x in class_ratios.tolist()]}")
print(f"Class weights for CE: {[round(x, 4) for x in class_weights.tolist()]}")
print(f"Adjusted loss weights: {[round(x, 4) for x in loss_class_weights.tolist()]}")



In [ ]:
class DiceCELoss(nn.Module):
    def __init__(
        self,
        num_classes=3,
        smooth=1.0,
        ce_weights=None,
        dice_weights=None,
        focal_gamma=0.0,
    ):
        super(DiceCELoss, self).__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ce_weights = ce_weights
        self.dice_weights = dice_weights
        self.focal_gamma = focal_gamma

    def forward(self, inputs, targets):
        targets = torchvision.transforms.functional.center_crop(targets, inputs.shape[-2:])

        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.ce_weights,
            reduction="none",
        )
        if self.focal_gamma > 0:
            pt = torch.exp(-ce_loss)
            ce_loss = ((1.0 - pt) ** self.focal_gamma) * ce_loss
        ce_loss = ce_loss.mean()

        inputs = torch.softmax(inputs, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(
            targets, num_classes=self.num_classes
        ).permute(0, 3, 1, 2).float()

        intersection = (inputs * targets_one_hot).sum(dim=(2, 3))
        dice = (2.0 * intersection + self.smooth) / (
            inputs.sum(dim=(2, 3)) + targets_one_hot.sum(dim=(2, 3)) + self.smooth
        )
        if self.dice_weights is not None:
            weights = self.dice_weights.to(dice.device).view(1, -1)
            dice_loss = 1 - ((dice * weights).sum(dim=1) / weights.sum(dim=1)).mean()
        else:
            dice_loss = 1 - dice.mean()

        return ce_loss + dice_loss


def mean_iou(pred, mask, num_classes=3):
    mask = torchvision.transforms.functional.center_crop(mask, pred.shape[-2:])
    ious = []
    pred = torch.argmax(pred, dim=1).reshape(-1)
    mask = mask.reshape(-1)

    for cls in range(num_classes):
        pred_cls = pred == cls
        mask_cls = mask == cls
        intersection = (pred_cls & mask_cls).sum().float()
        union = (pred_cls | mask_cls).sum().float()

        if union == 0:
            ious.append(float("nan"))
        else:
            ious.append((intersection / union).item())

    return np.nanmean(ious)


def summarize_segmentation_metrics(
    class_intersections, class_unions, correct_pixels, total_pixels
):
    class_ious = []
    for cls in range(len(class_intersections)):
        if class_unions[cls] == 0:
            class_ious.append(float("nan"))
        else:
            class_ious.append((class_intersections[cls] / class_unions[cls]).item())

    pixel_acc = correct_pixels / max(total_pixels, 1)
    mean_class_iou = float(np.nanmean(class_ious))
    return pixel_acc, mean_class_iou, class_ious


def evaluate_model(model, loader, criterion=None, num_classes=3):
    class_intersections = torch.zeros(num_classes, dtype=torch.float64, device=device)
    class_unions = torch.zeros(num_classes, dtype=torch.float64, device=device)
    total_correct = 0
    total_pixels = 0
    total_loss = 0.0

    model.eval()
    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(device, non_blocking=pin_memory)
            masks = masks.to(device, non_blocking=pin_memory)
            outputs = model(images)
            masks = torchvision.transforms.functional.center_crop(masks, outputs.shape[-2:])
            if criterion is not None:
                total_loss += criterion(outputs, masks).item()
            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == masks).sum().item()
            total_pixels += masks.numel()

            for cls in range(num_classes):
                pred_cls = preds == cls
                mask_cls = masks == cls
                class_intersections[cls] += (pred_cls & mask_cls).sum()
                class_unions[cls] += (pred_cls | mask_cls).sum()

    pixel_acc, mean_class_iou, class_ious = summarize_segmentation_metrics(
        class_intersections, class_unions, total_correct, total_pixels
    )
    avg_loss = total_loss / max(len(loader), 1) if criterion is not None else float("nan")

    return {
        "loss": avg_loss,
        "acc": pixel_acc,
        "miou": mean_class_iou,
        "class_ious": class_ious,
    }


In [ ]:
model = Unet(num_classes=3).to(device)
criterion = DiceCELoss(
    num_classes=3,
    ce_weights=loss_class_weights.to(device),
    dice_weights=None,
    focal_gamma=0.0,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1.5e-4
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
num_epochs = 100
early_stopping_patience = 12
early_stopping_min_delta = 0.0005

best_val_iou = 0.0
best_val_metrics = None
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "train_acc": [],
    "train_miou": [],
    "val_loss": [],
    "val_acc": [],
    "val_miou": [],
    "lr": [],
}

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    train_intersections = torch.zeros(3, dtype=torch.float64, device=device)
    train_unions = torch.zeros(3, dtype=torch.float64, device=device)
    train_correct = 0
    train_pixels = 0

    for images, masks in train_loader:
        images = images.to(device, non_blocking=pin_memory)
        masks = masks.to(device, non_blocking=pin_memory)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            masks = torchvision.transforms.functional.center_crop(masks, outputs.shape[-2:])
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        preds = torch.argmax(outputs.detach(), dim=1)
        train_correct += (preds == masks).sum().item()
        train_pixels += masks.numel()

        for cls in range(3):
            pred_cls = preds == cls
            mask_cls = masks == cls
            train_intersections[cls] += (pred_cls & mask_cls).sum()
            train_unions[cls] += (pred_cls | mask_cls).sum()

    avg_loss = epoch_loss / len(train_loader)
    train_acc, train_iou, train_class_ious = summarize_segmentation_metrics(
        train_intersections, train_unions, train_correct, train_pixels
    )
    val_metrics = evaluate_model(model, val_loader, criterion=criterion)
    scheduler.step(val_metrics["miou"])

    current_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)
    history["train_miou"].append(train_iou)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_miou"].append(val_metrics["miou"])
    history["lr"].append(current_lr)
    print(f"Epoch [{epoch}/{num_epochs}] | LR: {current_lr:.6f}")
    print(
        f"Train -> Loss: {avg_loss:.4f} | Acc: {train_acc:.4f} | IoU: {train_iou:.4f}"
    )
    print(
        f"Val   -> Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['acc']:.4f} | "
        f"IoU: {val_metrics['miou']:.4f}"
    )
    print(f"Val Class IOUs: {[round(x, 4) for x in val_metrics['class_ious']]}")

    if val_metrics["miou"] > best_val_iou + early_stopping_min_delta:
        best_val_iou = val_metrics["miou"]
        best_val_metrics = val_metrics.copy()
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print("--> Saved new best validation model!")
    else:
        epochs_without_improvement += 1
        print(
            f"    No validation improvement for {epochs_without_improvement}/"
            f"{early_stopping_patience} epochs."
        )

    if epochs_without_improvement >= early_stopping_patience:
        print("Early stopping triggered.")
        break

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
test_metrics = evaluate_model(model, test_loader, criterion=criterion)
if best_val_metrics is not None:
    print(
        f"Best Val -> Loss: {best_val_metrics['loss']:.4f} | "
        f"Acc: {best_val_metrics['acc']:.4f} | IoU: {best_val_metrics['miou']:.4f}"
    )
print(
    f"Test  -> Loss: {test_metrics['loss']:.4f} | "
    f"Acc: {test_metrics['acc']:.4f} | IoU: {test_metrics['miou']:.4f}"
)
print(f"Final Test Class IOUs: {[round(x, 4) for x in test_metrics['class_ious']]}")

HISTORY_PATH.write_text(json.dumps(history, indent=2))
print(f"Saved training history to {HISTORY_PATH}")


In [ ]:
if "history" not in globals():
    if not HISTORY_PATH.exists():
        raise FileNotFoundError(f"Training history not found: {HISTORY_PATH}")
    history = json.loads(HISTORY_PATH.read_text())

epochs = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(epochs, history["train_loss"], label="Train")
axes[0, 0].plot(epochs, history["val_loss"], label="Val")
axes[0, 0].set_title("Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend()

axes[0, 1].plot(epochs, history["train_acc"], label="Train")
axes[0, 1].plot(epochs, history["val_acc"], label="Val")
axes[0, 1].set_title("Pixel Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].legend()

axes[1, 0].plot(epochs, history["train_miou"], label="Train")
axes[1, 0].plot(epochs, history["val_miou"], label="Val")
axes[1, 0].set_title("Mean IoU")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("IoU")
axes[1, 0].legend()
1
axes[1, 1].plot(epochs, history["lr"], color="tab:green")
axes[1, 1].set_title("Learning Rate")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("LR")

fig.tight_layout()
fig.savefig(HISTORY_PLOT_PATH, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved training history plot to {HISTORY_PLOT_PATH}")



In [ ]:
def plot_and_save_results(model, dataset, num_samples=20):
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(f"Trained weights not found: {CHECKPOINT_PATH}")

    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
    model.eval()

    SAMPLE_PLOT_DIR.mkdir(parents=True, exist_ok=True)

    total_iou = 0.0
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)

    print("Generating and saving plots...")
    for i, idx in enumerate(indices):
        img_tensor, mask_tensor = dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)

        with torch.inference_mode():
            output = model(input_tensor)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

        total_iou += mean_iou(output, mask_tensor.unsqueeze(0).to(device))

        cropped_img = torchvision.transforms.functional.center_crop(img_tensor, pred_mask.shape)
        cropped_mask = torchvision.transforms.functional.center_crop(mask_tensor, pred_mask.shape)
        rgb_img = cropped_img.permute(1, 2, 0).numpy()
        gt_mask = cropped_mask.numpy()

        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.imshow(rgb_img)
        plt.title("RGB Input (center crop)")
        plt.axis("off")

        plt.subplot(1, 3, 2)
        plt.imshow(pred_mask, vmin=0, vmax=2)
        plt.title("Predicted Mask")
        plt.axis("off")

        plt.subplot(1, 3, 3)
        plt.imshow(gt_mask, vmin=0, vmax=2)
        plt.title("Ground Truth (cropped)")
        plt.axis("off")

        plt.tight_layout()
        plt.savefig(SAMPLE_PLOT_DIR / f"sample_{i + 1}.png")
        plt.close()

    print(f"Successfully saved {len(indices)} plots to {SAMPLE_PLOT_DIR}")
    print(f"Final Average Mean IOU on sample set: {total_iou / len(indices):.4f}")


plot_and_save_results(model, test_dataset, num_samples=20)
